# 🥚 Huevos Sinifana SAS — análisis paso a paso

**6 lotes de gallina ponedora · ~6 años (2016–2022) · una granja en Caldas–Antioquia, Colombia.**

Caso real: a partir de registros de producción, facturas de alimento, salarios y precios de mercado, reconstruyo el **costo y el margen por huevo** de cada lote (G1–G6).

_Real case: from production records, feed invoices, wages and market prices, I rebuild each lot's **cost and margin per egg**._

> Las figuras de abajo están pre-generadas (ver `_make_figures.py`); los bloques de código reproducen el mismo análisis. / Figures are pre-rendered; the code cells reproduce the same analysis.

## 0 · Datos / Data

Tablas limpias en `../data/` (ver `DATA_DICTIONARY.md`). Todo en pesos colombianos (COP).

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

DATA = Path('..') / 'data'
prod = pd.read_csv(DATA / 'production_weekly.csv')   # postura semanal por lote
comp = pd.read_csv(DATA / 'lot_comparison.csv')      # costo y margen por lote
mkt  = pd.read_csv(DATA / 'egg_market_price.csv')    # precio mayorista SIPSA
feed = pd.read_csv(DATA / 'feed_price.csv')          # precio concentrado (real/estimado)

# paleta de marca Huevos del Vecino
BROWN, YOLK, RED, KRAFT, OLIVE = '#41322F', '#E5B441', '#AE4920', '#C9B79C', '#6E8B3F'
comp

## 1 · Producción — ¿cómo puso cada lote? / How did each lot lay?

Postura **real** vs. el **estándar** de cada genética. Cuando la real va pegada o por encima del estándar, el manejo es bueno.

![Curva de postura G5](figures/01_lay_curve_g5.png)

In [ ]:
d = prod[(prod.lot == 'G5') & (prod.eggs > 0)].sort_values('week')
plt.figure(figsize=(8, 3.6))
plt.plot(d.week, d.lay_rate_standard * 100, '--', color=KRAFT, label='Estándar / Standard')
plt.plot(d.week, d.lay_rate_real * 100, color=RED, lw=2.4, label='Real / Actual')
plt.title('G5 · Lohmann Brown — postura real vs estándar')
plt.xlabel('Semana de vida'); plt.ylabel('% postura'); plt.legend(frameon=False); plt.show()

## 2 · Costo por huevo (modelo de 7 rubros) / Cost per egg

El **alimento es ~76%** del costo. La **nómina y servicios** (costo fijo de la granja) se reparten entre la producción real de cada día, así que los lotes que corrieron al tiempo (G3/G4) **comparten** ese costo. **El tamaño del lote pesa:** G6 fue chico → la nómina se reparte entre menos huevos → el huevo más caro.

![Costo por huevo por lote](figures/02_cost_breakdown.png)

In [ ]:
lots = comp.lot.tolist()
plt.figure(figsize=(8, 3.8))
plt.bar(lots, comp.cost_feed, color=YOLK, label='Alimento / Feed')
plt.bar(lots, comp.cost_labor, bottom=comp.cost_feed, color=RED, label='Mano de obra / Labor')
plt.bar(lots, comp.cost_other, bottom=comp.cost_feed + comp.cost_labor, color=KRAFT, label='Otros / Other')
plt.title('Costo por huevo, por lote (COP)'); plt.ylabel('COP / huevo')
plt.legend(frameon=False, ncol=3); plt.show()

## 3 · El insumo: precio del concentrado / Feed price

El concentrado **casi se duplicó** en la ventana. Honestidad: **factura real de Contegral** donde existe (2020–2023, puntos marrones); antes, **estimado** con un modelo concentrado ≈ f(maíz, soya) calibrado contra facturas (R²=0.74, puntos claros).

![Precio del concentrado](figures/03_feed_price.png)

In [ ]:
f = feed.copy(); f['fecha'] = pd.to_datetime(f.month + '-01')
plt.figure(figsize=(8, 3.4))
plt.plot(f.fecha, f.price_per_kg, color=YOLK, lw=2)
inv = f[f.source == 'invoice']; est = f[f.source == 'estimated']
plt.scatter(inv.fecha, inv.price_per_kg, color=BROWN, s=14, zorder=3, label='Factura real')
plt.scatter(est.fecha, est.price_per_kg, color='#EBD9A6', s=14, zorder=3, label='Estimado')
plt.title('Precio del concentrado ponedora (COP/kg)'); plt.ylabel('COP / kg')
plt.legend(frameon=False); plt.show()
print('Variación %d%% entre %s y %s' % (round((f.price_per_kg.iloc[-1] / f.price_per_kg.iloc[0] - 1) * 100), f.month.iloc[0], f.month.iloc[-1]))

## 4 · Veredicto: margen por lote / The verdict

🟢 verde = margen > $30 · 🟡 amarillo = $0–30 · 🔴 rojo = pérdida. La gallina pone igual, pero el **margen lo decide el mercado** y la **escala** del lote.

![Margen por lote](figures/04_margin_by_lot.png)

In [ ]:
cmap = {'green': OLIVE, 'yellow': YOLK, 'red': RED}
plt.figure(figsize=(8, 3.4))
plt.bar(comp.lot, comp.margin, color=[cmap[s] for s in comp.status])
plt.axhline(0, color='#7A6A5A', lw=.9)
plt.title('Margen neto por huevo, por lote (COP)'); plt.ylabel('COP / huevo'); plt.show()

## Hallazgos / Findings

1. **El alimento manda** (~76% del costo). / Feed is ~76% of cost.
2. **El insumo se encareció** (~$1.200 → ~$2.150/kg). / Input prices rose.
3. **El tamaño del lote pesa** — G6 chico = huevo más caro ($408). / Lot size matters.
4. **El mercado decide el margen** — la gallina pone igual. / The market sets the margin.
5. **No todos dejaron plata** — de −$37 (G1) a +$65 (G6). / Not every lot made money.

### Honestidad / Transparency
- Producción: registros reales semanales. / Real weekly records.
- Costo fijo repartido entre lotes que se traslaparon (G3/G4). / Fixed cost shared across overlapping lots.
- Alimento: factura real donde existe; si no, estimado (R²=0.74). / Real invoices where available; else estimated.
- Mercado: SIPSA-DANE mayorista; productor = mayorista × ~0,89. G1 anterior a SIPSA (2018). / SIPSA wholesale; producer = wholesale × ~0.89.

---
**Andrés Estrada** · estrada0788@gmail.com · GitHub: AndresTheAnalyst · _Datos reales de Huevos Sinifana SAS, usados con permiso._